<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit0/w01-environment/notebook.ipynb)


# Unit 1 — The environment

**Week 0 · prerequisite · about 45 minutes**

**Goal:** Know which Python is running, why it has to be the one uv made, and how every notebook here finds the repo.

**Why it matters:** Roughly half of all setup failures in any course are one thing: the kernel is a different Python from the one the dependencies went into. Everything imports fine in the terminal and nothing imports in the notebook. Twenty minutes here saves that.

Some cells below ship **broken on purpose**, marked `<------ EDIT THIS LINE`. Run them first and
read what happens. Debugging something wrong teaches more than filling in a blank.

In [1]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = ollama (llama3.2:1b at http://localhost:11434/v1)
ready. LIVE is the ollama lane.


In [2]:
from bootcamp_agent.checks import check, review
from bootcamp_agent.hints import hint  # noqa: F401 - hint("w01-e2") when you want a nudge
import bootcamp_agent.week0_checks  # noqa: F401 — importing is what registers them

## 1. Which Python is this, really

**Context.** `uv sync` installed everything into `.venv` inside the repo. Your notebook kernel
may or may not be that Python. If it is not, imports fail in ways that look like missing
packages and are not.

**Instructions.**

1. Read the values from the interpreter running this cell, not from what you believe you installed.
2. `in_repo_venv` compares `sys.prefix` against the repo's `.venv`.
3. The check reads the same values itself, so it cannot be satisfied by copying somebody else's answer.

In [3]:
reading = {
    "python_version": f"{sys.version_info.major}.{sys.version_info.minor}",   # <------ EDIT: this is wrong. Ask the interpreter.
    "in_repo_venv": Path(sys.prefix).resolve() == (REPO_ROOT / ".venv").resolve(),       # <------ EDIT: do not assume. Compare sys.prefix.
    "why_it_matters": "No module named 'bootcamp_agent'",       # TODO(you): what breaks when the kernel is a different Python?
}
print(f"sys.version_info -> {sys.version_info.major}.{sys.version_info.minor}")
print(f"sys.prefix       -> {sys.prefix}")
print(f"repo .venv       -> {REPO_ROOT / '.venv'}")
for k, v in reading.items():
    print(f"  {k:16} {v}")

sys.version_info -> 3.11
sys.prefix       -> d:\Ia Bootcamp\dev3pack-cohort-2026-09\.venv
repo .venv       -> d:\Ia Bootcamp\dev3pack-cohort-2026-09\.venv
  python_version   3.11
  in_repo_venv     True
  why_it_matters   No module named 'bootcamp_agent'


**Expected output**

```
sys.version_info -> 3.11
sys.prefix       -> /.../Dev3Pack-bootcamp-AI-Engineering/.venv
✅ w01-e1 passed
```

In [4]:
check("w01-e1", reading)

✅ w01-e1 passed


True

## 2. Find the repo root from anywhere

**Context.** Every notebook in this course opens with the same four lines: climb the directory
tree until `pyproject.toml` appears. It works from any subdirectory, which is why you can open
a notebook from anywhere.

The stub below climbs, and never stops.

**Instructions.**

1. Fix it so it terminates when there is no `pyproject.toml` anywhere above.
2. `path.parent` of the filesystem root is the root itself. That is your stopping condition.

In [5]:
def locate_repo_root(start: Path) -> Path:
    here = Path(start).resolve()
    while not (here / "pyproject.toml").exists():
        if here == here.parent:
            return None
        here = here.parent  # <------ EDIT THIS: what happens at the filesystem root?
    return here


print(locate_repo_root(REPO_ROOT / "modules" / "module-0"))

D:\Ia Bootcamp\dev3pack-cohort-2026-09


**Expected output**

```
/home/you/Dev3Pack-bootcamp-AI-Engineering
✅ w01-e2 passed
```

In [6]:
check("w01-e2", locate_repo_root)

✅ w01-e2 passed


True

## 3. Runtime dependency, or developer tool

**Context.** `pyproject.toml` separates what a *user* of the package needs from what somebody
*working on it* needs. Get it wrong and you ship a test runner to production, or you ship a
package that cannot run.

**Instructions.** For each package, answer `"main"` or `"dev"`. Open `pyproject.toml` and read it.

In [7]:
groups = {
    "pytest": "dev",         # <------ EDIT: does a USER of this package need pytest?
    "ruff": "dev",           # <------ EDIT
    "nbformat": "dev",       # <------ EDIT
    "python-dotenv": "main",
}
for package, group in groups.items():
    print(f"  {package:16} {group}")

  pytest           dev
  ruff             dev
  nbformat         dev
  python-dotenv    main


**Expected output**

```
  pytest           dev
  ruff             dev
  nbformat         dev
  python-dotenv    main
✅ w01-e3 passed
```

In [8]:
check("w01-e3", groups)

✅ w01-e3 passed


True

## Review

The scorecard for this unit. Every ❌ line names the exercise and the hint.

In [9]:
review("w01")

w01: 3/3 passed  ·  300/300 marks


True